In [1]:
import os

import glob
import copy
import numpy as np
import scipy
import scipy.optimize
import scipy.interpolate
import astropy.table
import astropy.io.fits
import astropy.units
import yaml

import matplotlib.pyplot as plt
import matplotlib as mpl


import lezargus

# %matplotlib Qt

ToDoError - Flux conserving interpolation is needed to be implemented; defaulting to Linear.


ToDoError - Uncertainty values on integrations need to be done.


ToDoError - Uncertainty values on integrations need to be done.


ToDoError - Flux conserving interpolation is needed to be implemented; defaulting to Linear.


ToDoError - Uncertainty values on integrations need to be done.


ToDoError - Uncertainty values on integrations need to be done.


... (53 outputs left out)

AccuracyWarning - Approximating the arc lamp level as about the flat lamp level, multiplying by 1e-11


AccuracyWarning - Approximating the solution arc lamp level as about the simulation level, multiplying by 1


AccuracyWarning - Approximating the solution arc lamp level as about the simulation level, multiplying by 1


AccuracyWarning - Approximating the solution arc lamp level as about the simulation level, multiplying by 1


In [2]:
lezargus.initialize.initialize_logging_outputs()
# Making sure that the output directory exists.
os.makedirs("./products/spectre_detectors/", exist_ok=True)

# Visible CCD Detector

### Main Parameters

In [3]:
visible_detector = {
    "detector_shape": [2048, 2048],
    "pixel_size": 15 * 1e-6,
    "gain": 6.95,
    "nondestructive_reads": 1,
    "read_noise_mean": 3,
    "read_noise_stddev": 15,
    "hot_pixel_value": +np.inf,
    "dead_pixel_value": 0,
    "cosmic_ray_rate": 0,
    "cosmic_ray_value": 1,
}

# Saving these main parameters just as a simple text dictionary/yaml.
visible_detector_filename = (
    f"./products/spectre_detectors/visible_detector_properties.yaml"
)
with open(visible_detector_filename, "wt", encoding="utf8") as visible_file:
    yaml.dump(visible_detector, visible_file, default_flow_style=False)

### Bias Frames

In [4]:
# Some dummy Gaussian-based bias frames because we don't have real data.
bias_gaussian_mean = 5
bias_gaussian_stddev = 1

generator = np.random.default_rng()

# Generating a few files...
random_visible_bias_frames = []
for index, __ in enumerate(range(10)):
    # Random numbers.
    random_frame = generator.normal(
        loc=bias_gaussian_mean,
        scale=bias_gaussian_stddev,
        size=visible_detector["detector_shape"],
    )
    # Can't have negative bias.
    random_frame[random_frame <= 0] = 0
    random_visible_bias_frames.append(random_frame)

# Average and stddev for the frames.
visible_bias_frame_mean = np.mean(random_visible_bias_frames, axis=0)
visible_bias_frame_stddev = np.std(random_visible_bias_frames, axis=0)

# Creating the filename for this bias frame.
mean_filename = f"./products/spectre_detectors/visible_bias_frame_mean.npy"
stddev_filename = f"./products/spectre_detectors/visible_bias_frame_stddev.npy"
mean_saving_array = np.array(visible_bias_frame_mean, dtype=np.half)
stddev_saving_array = np.array(visible_bias_frame_stddev, dtype=np.half)
np.save(mean_filename, mean_saving_array, allow_pickle=False)
np.save(stddev_filename, stddev_saving_array, allow_pickle=False)

### Flat Frames

In [5]:
# Some dummy Gaussian-based flat frames because we don't have real data.
flat_gaussian_mean = 1
flat_gaussian_stddev = 0.05

generator = np.random.default_rng()

# Generating a few files...
random_visible_flat_frames = []
for index, __ in enumerate(range(10)):
    # Random numbers.
    random_frame = generator.normal(
        loc=flat_gaussian_mean,
        scale=flat_gaussian_stddev,
        size=visible_detector["detector_shape"],
    )
    # Can't have negative bias.
    random_frame[random_frame <= 0] = 0
    random_visible_flat_frames.append(random_frame)

# Average and stddev for the frames.
visible_flat_frame_mean = np.mean(random_visible_flat_frames, axis=0)
visible_flat_frame_stddev = np.std(random_visible_flat_frames, axis=0)

# Creating the filename for this bias frame.
mean_filename = f"./products/spectre_detectors/visible_flat_frame_mean.npy"
stddev_filename = f"./products/spectre_detectors/visible_flat_frame_stddev.npy"
mean_saving_array = np.array(visible_flat_frame_mean, dtype=np.half)
stddev_saving_array = np.array(visible_flat_frame_stddev, dtype=np.half)
np.save(mean_filename, mean_saving_array, allow_pickle=False)
np.save(stddev_filename, stddev_saving_array, allow_pickle=False)

### Dark Current

In [6]:
# Some dummy Gaussian-based dark current frames because we don't have real data.
dark_curr_gaussian_mean = 1
dark_curr_gaussian_stddev = 1

generator = np.random.default_rng()

# Generating a few files...
random_visible_dark_current_frames = []
for index, __ in enumerate(range(10)):
    # Random numbers.
    random_frame = generator.normal(
        loc=dark_curr_gaussian_mean,
        scale=dark_curr_gaussian_stddev,
        size=visible_detector["detector_shape"],
    )
    # Can't have negative bias.
    random_frame[random_frame <= 0] = 0
    random_visible_dark_current_frames.append(random_frame)

# Average and stddev for the frames.
visible_dark_current_frame_mean = np.mean(
    random_visible_dark_current_frames, axis=0
)
visible_dark_current_frame_stddev = np.std(
    random_visible_dark_current_frames, axis=0
)

# Creating the filename for this bias frame.
mean_filename = (
    f"./products/spectre_detectors/visible_dark_current_frame_mean.npy"
)
stddev_filename = (
    f"./products/spectre_detectors/visible_dark_current_frame_stddev.npy"
)
mean_saving_array = np.array(visible_dark_current_frame_mean, dtype=np.half)
stddev_saving_array = np.array(visible_dark_current_frame_stddev, dtype=np.half)
np.save(mean_filename, mean_saving_array, allow_pickle=False)
np.save(stddev_filename, stddev_saving_array, allow_pickle=False)

### Linearity Function

In [7]:
# The linearity function of the detector.
visible_linearity_input_name = (
    "./base/spectre_detectors/spectre_visible_linearity.dat"
)
visible_linearity_data = np.genfromtxt(
    visible_linearity_input_name, dtype=float, comments="#", delimiter="|"
)
visible_linearity_input, visible_linearity_output = np.transpose(
    visible_linearity_data
)

# We package the data into plain-text MRT tables. The actual meta data needs
# to be added later.
# Packaging...
linearity_input_column = astropy.table.Column(
    visible_linearity_input, name="input", unit="count", dtype=float
)
linearity_output_column = astropy.table.Column(
    visible_linearity_output, name="output", unit="count", dtype=float
)

# Readout.
readout_table_columns = [linearity_input_column, linearity_output_column]
visible_linearity_table = astropy.table.Table(readout_table_columns)
# Too much precision makes unnessary large file sizes.
visible_linearity_filename = (
    f"./products/spectre_detectors/visible_linearity_function.dat"
)
visible_linearity_table.write(
    visible_linearity_filename,
    format="ascii.mrt",
    formats={keydex: "%.3e" for keydex in visible_linearity_table.keys()},
    overwrite=True,
)

### Quantum Efficiency Function

In [8]:
# The quantum efficiency (QE) of the detector.
visible_ccd_qe_input_name = (
    "./base/spectre_detectors/spectre_visible_ccd_qe.dat"
)
visible_ccd_qe_data = np.genfromtxt(
    visible_ccd_qe_input_name, dtype=float, comments="#", delimiter="|"
)
visible_ccd_qe_wavelength, visible_ccd_qe_efficiency = np.transpose(
    visible_ccd_qe_data
)

# We package the data into plain-text MRT tables. The actual meta data needs
# to be added later.
# Converting...
wavelength_meter = lezargus.library.conversion.convert_units(
    visible_ccd_qe_wavelength, value_unit="um", result_unit="m"
)
efficiency_unitless = visible_ccd_qe_efficiency
# Packaging...
wavelength_column = astropy.table.Column(
    wavelength_meter, name="wavelength", unit="m", dtype=float
)
data_column = astropy.table.Column(
    efficiency_unitless, name="efficiency", unit="", dtype=float
)

# Readout.
readout_table_columns = [wavelength_column, data_column]
visible_ccd_qe_table = astropy.table.Table(readout_table_columns)
# Too much precision makes unnessary large file sizes.
visible_ccd_qe_filename = (
    f"./products/spectre_detectors/visible_ccd_quantum_efficiency.dat"
)
visible_ccd_qe_table.write(
    visible_ccd_qe_filename,
    format="ascii.mrt",
    formats={keydex: "%.3e" for keydex in visible_ccd_qe_table.keys()},
    overwrite=True,
)

### Hot Pixel Map

In [9]:
# Some dummy data modeling a couple of hot pixels because we do not have
# real data.
hot_pixel_probability = 0.0001

generator = np.random.default_rng()

# Random numbers.
random_visible_hot_pixel_criterion = generator.random(
    size=visible_detector["detector_shape"]
)
visible_hot_pixel_map = (
    random_visible_hot_pixel_criterion <= hot_pixel_probability
)

# Creating the filename for this hot pixel map.
filename = f"./products/spectre_detectors/visible_hot_pixel_map.npy"
saving_array = np.array(visible_hot_pixel_map, dtype=np.bool)
np.save(filename, saving_array, allow_pickle=False)

### Dead Pixel Map

In [10]:
# Some dummy data modeling a couple of dead pixels because we do not have
# real data.
dead_pixel_probability = 0.0001

generator = np.random.default_rng()

# Random numbers.
random_visible_dead_pixel_criterion = generator.random(
    size=visible_detector["detector_shape"]
)
visible_dead_pixel_map = (
    random_visible_dead_pixel_criterion <= dead_pixel_probability
)

# Creating the filename for this hot pixel map.
filename = f"./products/spectre_detectors/visible_dead_pixel_map.npy"
saving_array = np.array(visible_dead_pixel_map, dtype=np.bool)
np.save(filename, saving_array, allow_pickle=False)

# NearIR H2RG 25 Detector

### Main Parameters

In [11]:
nearir_detector = {
    "detector_shape": [2048, 2048],
    "pixel_size": 18 * 1e-6,
    "gain": 6.95,
    "nondestructive_reads": 32,
    "read_noise_mean": 3,
    "read_noise_stddev": 15,
    "hot_pixel_value": +np.inf,
    "dead_pixel_value": 0,
    "cosmic_ray_rate": 0,
    "cosmic_ray_value": 1,
}

# Saving these main parameters just as a simple text dictionary/yaml.
nearir_detector_filename = (
    f"./products/spectre_detectors/nearir_detector_properties.yaml"
)
with open(nearir_detector_filename, "wt", encoding="utf8") as nearir_file:
    yaml.dump(nearir_detector, nearir_file, default_flow_style=False)

### Bias Frames

In [12]:
# Some dummy Gaussian-based bias frames because we don't have real data.
bias_gaussian_mean = 5
bias_gaussian_stddev = 1

generator = np.random.default_rng()

# Generating a few files...
random_nearir_bias_frames = []
for index, __ in enumerate(range(10)):
    # Random numbers.
    random_frame = generator.normal(
        loc=bias_gaussian_mean,
        scale=bias_gaussian_stddev,
        size=nearir_detector["detector_shape"],
    )
    # Can't have negative bias.
    random_frame[random_frame <= 0] = 0
    random_nearir_bias_frames.append(random_frame)

# Average and stddev for the frames.
nearir_bias_frame_mean = np.mean(random_nearir_bias_frames, axis=0)
nearir_bias_frame_stddev = np.std(random_nearir_bias_frames, axis=0)

# Creating the filename for this bias frame.
mean_filename = f"./products/spectre_detectors/nearir_bias_frame_mean.npy"
stddev_filename = f"./products/spectre_detectors/nearir_bias_frame_stddev.npy"
mean_saving_array = np.array(nearir_bias_frame_mean, dtype=np.half)
stddev_saving_array = np.array(nearir_bias_frame_stddev, dtype=np.half)
np.save(mean_filename, mean_saving_array, allow_pickle=False)
np.save(stddev_filename, stddev_saving_array, allow_pickle=False)

### Flat Frames

In [13]:
# Some dummy Gaussian-based flat frames because we don't have real data.
flat_gaussian_mean = 1
flat_gaussian_stddev = 0.05

generator = np.random.default_rng()

# Generating a few files...
random_nearir_flat_frames = []
for index, __ in enumerate(range(10)):
    # Random numbers.
    random_frame = generator.normal(
        loc=flat_gaussian_mean,
        scale=flat_gaussian_stddev,
        size=nearir_detector["detector_shape"],
    )
    # Can't have negative bias.
    random_frame[random_frame <= 0] = 0
    random_nearir_flat_frames.append(random_frame)

# Average and stddev for the frames.
nearir_flat_frame_mean = np.mean(random_nearir_flat_frames, axis=0)
nearir_flat_frame_stddev = np.std(random_nearir_flat_frames, axis=0)

# Creating the filename for this bias frame.
mean_filename = f"./products/spectre_detectors/nearir_flat_frame_mean.npy"
stddev_filename = f"./products/spectre_detectors/nearir_flat_frame_stddev.npy"
mean_saving_array = np.array(nearir_flat_frame_mean, dtype=np.half)
stddev_saving_array = np.array(nearir_flat_frame_stddev, dtype=np.half)
np.save(mean_filename, mean_saving_array, allow_pickle=False)
np.save(stddev_filename, stddev_saving_array, allow_pickle=False)

### Dark Current

In [14]:
# Some dummy Gaussian-based dark current frames because we don't have real data.
dark_curr_gaussian_mean = 1
dark_curr_gaussian_stddev = 1

generator = np.random.default_rng()

# Generating a few files...
random_nearir_dark_current_frames = []
for index, __ in enumerate(range(10)):
    # Random numbers.
    random_frame = generator.normal(
        loc=dark_curr_gaussian_mean,
        scale=dark_curr_gaussian_stddev,
        size=nearir_detector["detector_shape"],
    )
    # Can't have negative bias.
    random_frame[random_frame <= 0] = 0
    random_nearir_dark_current_frames.append(random_frame)

# Average and stddev for the frames.
nearir_dark_current_frame_mean = np.mean(
    random_nearir_dark_current_frames, axis=0
)
nearir_dark_current_frame_stddev = np.std(
    random_nearir_dark_current_frames, axis=0
)

# Creating the filename for this bias frame.
mean_filename = (
    f"./products/spectre_detectors/nearir_dark_current_frame_mean.npy"
)
stddev_filename = (
    f"./products/spectre_detectors/nearir_dark_current_frame_stddev.npy"
)
mean_saving_array = np.array(nearir_dark_current_frame_mean, dtype=np.half)
stddev_saving_array = np.array(nearir_dark_current_frame_stddev, dtype=np.half)
np.save(mean_filename, mean_saving_array, allow_pickle=False)
np.save(stddev_filename, stddev_saving_array, allow_pickle=False)

### Linearity Function

In [15]:
# The linearity function of the detector.
nearir_linearity_input_name = (
    "./base/spectre_detectors/spectre_nearir_linearity.dat"
)
nearir_linearity_data = np.genfromtxt(
    nearir_linearity_input_name, dtype=float, comments="#", delimiter="|"
)
nearir_linearity_input, nearir_linearity_output = np.transpose(
    nearir_linearity_data
)

# We package the data into plain-text MRT tables. The actual meta data needs
# to be added later.
# Packaging...
linearity_input_column = astropy.table.Column(
    nearir_linearity_input, name="input", unit="count", dtype=float
)
linearity_output_column = astropy.table.Column(
    nearir_linearity_output, name="output", unit="count", dtype=float
)

# Readout.
readout_table_columns = [linearity_input_column, linearity_output_column]
nearir_linearity_table = astropy.table.Table(readout_table_columns)
# Too much precision makes unnessary large file sizes.
nearir_linearity_filename = (
    f"./products/spectre_detectors/nearir_linearity_function.dat"
)
nearir_linearity_table.write(
    nearir_linearity_filename,
    format="ascii.mrt",
    formats={keydex: "%.3e" for keydex in nearir_linearity_table.keys()},
    overwrite=True,
)

### Quantum Efficiency Function

In [16]:
# The quantum efficiency (QE) of the detector.
nearir_ccd_qe_input_name = "./base/spectre_detectors/spectre_nearir_h2rg_qe.dat"
nearir_ccd_qe_data = np.genfromtxt(
    nearir_ccd_qe_input_name, dtype=float, comments="#", delimiter="|"
)
nearir_ccd_qe_wavelength, nearir_ccd_qe_efficiency = np.transpose(
    nearir_ccd_qe_data
)

# We package the data into plain-text MRT tables. The actual meta data needs
# to be added later.
# Converting...
wavelength_meter = lezargus.library.conversion.convert_units(
    nearir_ccd_qe_wavelength, value_unit="um", result_unit="m"
)
efficiency_unitless = nearir_ccd_qe_efficiency
# Packaging...
wavelength_column = astropy.table.Column(
    wavelength_meter, name="wavelength", unit="m", dtype=float
)
data_column = astropy.table.Column(
    efficiency_unitless, name="efficiency", unit="", dtype=float
)

# Readout.
readout_table_columns = [wavelength_column, data_column]
nearir_ccd_qe_table = astropy.table.Table(readout_table_columns)
# Too much precision makes unnessary large file sizes.
nearir_ccd_qe_filename = (
    f"./products/spectre_detectors/nearir_h2rg_quantum_efficiency.dat"
)
nearir_ccd_qe_table.write(
    nearir_ccd_qe_filename,
    format="ascii.mrt",
    formats={keydex: "%.3e" for keydex in nearir_ccd_qe_table.keys()},
    overwrite=True,
)

### Hot Pixel Map

In [17]:
# Some dummy data modeling a couple of hot pixels because we do not have
# real data.
hot_pixel_probability = 0.0001

generator = np.random.default_rng()

# Random numbers.
random_nearir_hot_pixel_criterion = generator.random(
    size=nearir_detector["detector_shape"]
)
nearir_hot_pixel_map = (
    random_nearir_hot_pixel_criterion <= hot_pixel_probability
)

# Creating the filename for this hot pixel map.
filename = f"./products/spectre_detectors/nearir_hot_pixel_map.npy"
saving_array = np.array(nearir_hot_pixel_map, dtype=np.bool)
np.save(filename, saving_array, allow_pickle=False)

### Dead Pixel Map

In [18]:
# Some dummy data modeling a couple of dead pixels because we do not have
# real data.
dead_pixel_probability = 0.0001

generator = np.random.default_rng()

# Random numbers.
random_nearir_dead_pixel_criterion = generator.random(
    size=nearir_detector["detector_shape"]
)
nearir_dead_pixel_map = (
    random_nearir_dead_pixel_criterion <= dead_pixel_probability
)

# Creating the filename for this hot pixel map.
filename = f"./products/spectre_detectors/nearir_dead_pixel_map.npy"
saving_array = np.array(nearir_dead_pixel_map, dtype=np.bool)
np.save(filename, saving_array, allow_pickle=False)

# MidIR H2RG 50 Detector

### Main Parameters

In [19]:
midir_detector = {
    "detector_shape": [2048, 2048],
    "pixel_size": 18 * 1e-6,
    "gain": 6.95,
    "nondestructive_reads": 32,
    "read_noise_mean": 3,
    "read_noise_stddev": 15,
    "hot_pixel_value": +np.inf,
    "dead_pixel_value": 0,
    "cosmic_ray_rate": 0,
    "cosmic_ray_value": 1,
}

# Saving these main parameters just as a simple text dictionary/yaml.
midir_detector_filename = (
    f"./products/spectre_detectors/midir_detector_properties.yaml"
)
with open(midir_detector_filename, "wt", encoding="utf8") as midir_file:
    yaml.dump(midir_detector, midir_file, default_flow_style=False)

### Bias Frames

In [20]:
# Some dummy Gaussian-based bias frames because we don't have real data.
bias_gaussian_mean = 5
bias_gaussian_stddev = 1

generator = np.random.default_rng()

# Generating a few files...
random_midir_bias_frames = []
for index, __ in enumerate(range(10)):
    # Random numbers.
    random_frame = generator.normal(
        loc=bias_gaussian_mean,
        scale=bias_gaussian_stddev,
        size=midir_detector["detector_shape"],
    )
    # Can't have negative bias.
    random_frame[random_frame <= 0] = 0
    random_midir_bias_frames.append(random_frame)

# Average and stddev for the frames.
midir_bias_frame_mean = np.mean(random_midir_bias_frames, axis=0)
midir_bias_frame_stddev = np.std(random_midir_bias_frames, axis=0)

# Creating the filename for this bias frame.
mean_filename = f"./products/spectre_detectors/midir_bias_frame_mean.npy"
stddev_filename = f"./products/spectre_detectors/midir_bias_frame_stddev.npy"
mean_saving_array = np.array(midir_bias_frame_mean, dtype=np.half)
stddev_saving_array = np.array(midir_bias_frame_stddev, dtype=np.half)
np.save(mean_filename, mean_saving_array, allow_pickle=False)
np.save(stddev_filename, stddev_saving_array, allow_pickle=False)

### Flat Frames

In [21]:
# Some dummy Gaussian-based flat frames because we don't have real data.
flat_gaussian_mean = 1
flat_gaussian_stddev = 0.05

generator = np.random.default_rng()

# Generating a few files...
random_midir_flat_frames = []
for index, __ in enumerate(range(10)):
    # Random numbers.
    random_frame = generator.normal(
        loc=flat_gaussian_mean,
        scale=flat_gaussian_stddev,
        size=midir_detector["detector_shape"],
    )
    # Can't have negative bias.
    random_frame[random_frame <= 0] = 0
    random_midir_flat_frames.append(random_frame)

# Average and stddev for the frames.
midir_flat_frame_mean = np.mean(random_midir_flat_frames, axis=0)
midir_flat_frame_stddev = np.std(random_midir_flat_frames, axis=0)

# Creating the filename for this bias frame.
mean_filename = f"./products/spectre_detectors/midir_flat_frame_mean.npy"
stddev_filename = f"./products/spectre_detectors/midir_flat_frame_stddev.npy"
mean_saving_array = np.array(midir_flat_frame_mean, dtype=np.half)
stddev_saving_array = np.array(midir_flat_frame_stddev, dtype=np.half)
np.save(mean_filename, mean_saving_array, allow_pickle=False)
np.save(stddev_filename, stddev_saving_array, allow_pickle=False)

### Dark Current

In [22]:
# Some dummy Gaussian-based dark current frames because we don't have real data.
dark_curr_gaussian_mean = 1
dark_curr_gaussian_stddev = 1

generator = np.random.default_rng()

# Generating a few files...
random_midir_dark_current_frames = []
for index, __ in enumerate(range(10)):
    # Random numbers.
    random_frame = generator.normal(
        loc=dark_curr_gaussian_mean,
        scale=dark_curr_gaussian_stddev,
        size=midir_detector["detector_shape"],
    )
    # Can't have negative bias.
    random_frame[random_frame <= 0] = 0
    random_midir_dark_current_frames.append(random_frame)

# Average and stddev for the frames.
midir_dark_current_frame_mean = np.mean(
    random_midir_dark_current_frames, axis=0
)
midir_dark_current_frame_stddev = np.std(
    random_midir_dark_current_frames, axis=0
)

# Creating the filename for this bias frame.
mean_filename = (
    f"./products/spectre_detectors/midir_dark_current_frame_mean.npy"
)
stddev_filename = (
    f"./products/spectre_detectors/midir_dark_current_frame_stddev.npy"
)
mean_saving_array = np.array(midir_dark_current_frame_mean, dtype=np.half)
stddev_saving_array = np.array(midir_dark_current_frame_stddev, dtype=np.half)
np.save(mean_filename, mean_saving_array, allow_pickle=False)
np.save(stddev_filename, stddev_saving_array, allow_pickle=False)

### Linearity Function

In [23]:
# The linearity function of the detector.
midir_linearity_input_name = (
    "./base/spectre_detectors/spectre_midir_linearity.dat"
)
midir_linearity_data = np.genfromtxt(
    midir_linearity_input_name, dtype=float, comments="#", delimiter="|"
)
midir_linearity_input, midir_linearity_output = np.transpose(
    midir_linearity_data
)

# We package the data into plain-text MRT tables. The actual meta data needs
# to be added later.
# Packaging...
linearity_input_column = astropy.table.Column(
    midir_linearity_input, name="input", unit="count", dtype=float
)
linearity_output_column = astropy.table.Column(
    midir_linearity_output, name="output", unit="count", dtype=float
)

# Readout.
readout_table_columns = [linearity_input_column, linearity_output_column]
midir_linearity_table = astropy.table.Table(readout_table_columns)
# Too much precision makes unnessary large file sizes.
midir_linearity_filename = (
    f"./products/spectre_detectors/midir_linearity_function.dat"
)
midir_linearity_table.write(
    midir_linearity_filename,
    format="ascii.mrt",
    formats={keydex: "%.3e" for keydex in midir_linearity_table.keys()},
    overwrite=True,
)

### Quantum Efficiency Function

In [24]:
# The quantum efficiency (QE) of the detector.
midir_ccd_qe_input_name = "./base/spectre_detectors/spectre_midir_h2rg_qe.dat"
midir_ccd_qe_data = np.genfromtxt(
    midir_ccd_qe_input_name, dtype=float, comments="#", delimiter="|"
)
midir_ccd_qe_wavelength, midir_ccd_qe_efficiency = np.transpose(
    midir_ccd_qe_data
)

# We package the data into plain-text MRT tables. The actual meta data needs
# to be added later.
# Converting...
wavelength_meter = lezargus.library.conversion.convert_units(
    midir_ccd_qe_wavelength, value_unit="um", result_unit="m"
)
efficiency_unitless = midir_ccd_qe_efficiency
# Packaging...
wavelength_column = astropy.table.Column(
    wavelength_meter, name="wavelength", unit="m", dtype=float
)
data_column = astropy.table.Column(
    efficiency_unitless, name="efficiency", unit="", dtype=float
)

# Readout.
readout_table_columns = [wavelength_column, data_column]
midir_ccd_qe_table = astropy.table.Table(readout_table_columns)
# Too much precision makes unnessary large file sizes.
midir_ccd_qe_filename = (
    f"./products/spectre_detectors/midir_h2rg_quantum_efficiency.dat"
)
midir_ccd_qe_table.write(
    midir_ccd_qe_filename,
    format="ascii.mrt",
    formats={keydex: "%.3e" for keydex in midir_ccd_qe_table.keys()},
    overwrite=True,
)

### Hot Pixel Map

In [25]:
# Some dummy data modeling a couple of hot pixels because we do not have
# real data.
hot_pixel_probability = 0.0001

generator = np.random.default_rng()

# Random numbers.
random_midir_hot_pixel_criterion = generator.random(
    size=midir_detector["detector_shape"]
)
midir_hot_pixel_map = random_midir_hot_pixel_criterion <= hot_pixel_probability

# Creating the filename for this hot pixel map.
filename = f"./products/spectre_detectors/midir_hot_pixel_map.npy"
saving_array = np.array(midir_hot_pixel_map, dtype=np.bool)
np.save(filename, saving_array, allow_pickle=False)

### Dead Pixel Map

In [26]:
# Some dummy data modeling a couple of dead pixels because we do not have
# real data.
dead_pixel_probability = 0.0001

generator = np.random.default_rng()

# Random numbers.
random_midir_dead_pixel_criterion = generator.random(
    size=midir_detector["detector_shape"]
)
midir_dead_pixel_map = (
    random_midir_dead_pixel_criterion <= dead_pixel_probability
)

# Creating the filename for this hot pixel map.
filename = f"./products/spectre_detectors/midir_dead_pixel_map.npy"
saving_array = np.array(midir_dead_pixel_map, dtype=np.bool)
np.save(filename, saving_array, allow_pickle=False)